# Rectified-CFG++ NeurIPS 2025 paper reproduction: SD3 / T2I-CompBench / configurable seed

This notebook has one purpose: reproduce the paper-reported SD3 comparison between **standard CFG** and **Rectified-CFG++**, using the authors' released implementation and the official T2I-CompBench scorers. No AIM-Flow/SPFC method is generated or scored.

## Locked comparison

| Item | CFG | Rectified-CFG++ |
|---|---:|---:|
| Guidance parameter | $\omega=4.5$ | $\lambda=7.0$ |
| SD3 checkpoint | `stabilityai/stable-diffusion-3-medium-diffusers` | same |
| Resolution / steps | 1024×1024 / 28 | same |
| Conditioning | CLIP-L + CLIP-G + T5-XXL | identical cached tensors |
| Negative prompt | empty | empty |
| Images per prompt | 10, seeds `RUN_SEED` through `RUN_SEED + 9` | same initial seeds |

The $\omega=4.5,\lambda=7.0$ pairing is the SD3 28-NFE comparison reported in Table 10 of the Rectified-CFG++ paper. The Rectified arm is loaded directly from the authors' repository pinned at commit `3c838882f7b2cdc2e1c3e5785468cdcb1fb27190`; setup clones/fetches it if necessary and verifies the exact pipeline hash.

Set `RUN_SEED` in the first code cell, then run the notebook from the top. The test set is a deterministic `random.Random(RUN_SEED)` sample of **100 prompts total**: 25 each from the complete official color, texture, spatial, and shape files. The same value starts the ten image seeds (`RUN_SEED` through `RUN_SEED + 9`). Ten images per prompt give **1,000 images per method** and **2,000 total**. Report these as seeded 100-prompt subset results, not full-set leaderboard results.

### Reproduction scope

This reproduces the paper's SD3 scale pairing using the authors' released executable pipeline. The paper does not disclose the exact seed, $\mu$, or complete T2I-CompBench Table 2 hyperparameter record, and Algorithm 1 is not textually identical to the released pipeline. The notebook records an audit of the code that actually runs instead of claiming stronger compliance than the released artifacts support.

Full CLIP-L/CLIP-G/T5-XXL conditioning is encoded with singleton topology, cached losslessly, and supplied identically to both methods. Generation is resumable; both methods run sequentially on the same physical GPU to avoid a method/hardware confound. The second GPU is used only to shard T5-XXL during conditioning-cache construction. Official BLIP-VQA scores color/texture/shape and official UniDet RS200 scores spatial relations.

Artifacts go to `outputs/rectified_cfgpp_paper_reproduction_t2i_100_seed{RUN_SEED}` unless `AIM_FLOW_RECTCFGPP_REPRO_ROOT` is set. Each seed therefore has an isolated, resumable output directory.


In [1]:
# USER CONFIGURATION: edit this value, then choose Run All.
# It controls both prompt-subset selection and the first of 10 generation seeds.
RUN_SEED = 13

if isinstance(RUN_SEED, bool) or not isinstance(RUN_SEED, int):
    raise TypeError("RUN_SEED must be an integer")
if not 0 <= RUN_SEED <= 4_294_967_295:
    raise ValueError("RUN_SEED must be between 0 and 4,294,967,295")
print(f"Selected run seed: {RUN_SEED}; generation seeds: {RUN_SEED}–{RUN_SEED + 9}")


Selected run seed: 13; generation seeds: 13–22


In [2]:
# Install the generation environment and clone/verify official repositories.
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "aim_flow").is_dir():
            return candidate.resolve()
    raise FileNotFoundError("Launch this notebook from inside the aim-flow repository.")


REPO_ROOT = find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)
RUN_INSTALLS = os.environ.get("AIM_FLOW_SKIP_INSTALLS", "0") != "1"

PINS = {
    "rectified_cfgpp": (
        "https://github.com/shreshthsaini/Rectified-CFGpp.git",
        "3c838882f7b2cdc2e1c3e5785468cdcb1fb27190",
    ),
    "t2i_compbench": (
        "https://github.com/Karine-Huang/T2I-CompBench.git",
        "1b7094991a57f3c22abdd4f6e8ba6c1a15517073",
    ),
    "openai_clip": (
        "https://github.com/openai/CLIP.git",
        "d05afc436d78f1c48dc0dbf8e5980a9d471f35f6",
    ),
    "detectron2": (
        "https://github.com/facebookresearch/detectron2.git",
        "5aeb252b194b93dc2879b4ac34bc51a31b5aee13",
    ),
}
GENERATION_PACKAGES = [
    "torch==2.10.0", "torchvision==0.25.0", "diffusers==0.30.2",
    "transformers==4.49.0", "accelerate==1.13.0", "huggingface-hub==0.29.3",
    "safetensors==0.7.0", "sentencepiece==0.2.1", "protobuf==5.29.3",
    "numpy==2.4.6", "Pillow==12.1.1", "pandas==3.0.0", "tqdm==4.67.1",
    "PyYAML==6.0.3", "psutil==7.0.0", "nbformat==5.10.4",
]


def run(command, *, cwd: Path | None = None, env=None, capture: bool = False):
    command = [str(x) for x in command]
    print("+", " ".join(command))
    return subprocess.run(
        command, cwd=str(cwd or REPO_ROOT), env=env, check=True,
        text=True, capture_output=capture,
    )


if RUN_INSTALLS:
    run([sys.executable, "-m", "pip", "install", "--upgrade", "pip==24.3.1", "setuptools==75.6.0", "wheel==0.45.1"])
    run([sys.executable, "-m", "pip", "install", *GENERATION_PACKAGES])
    run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(REPO_ROOT)])

EXTERNAL = REPO_ROOT / "external"
REPO_PATHS = {
    "rectified_cfgpp": EXTERNAL / "Rectified-CFGpp",
    "t2i_compbench": EXTERNAL / "T2I-CompBench",
}


def git_output(repo: Path, *args: str) -> str:
    return subprocess.check_output(["git", "-C", str(repo), *args], text=True).strip()


def ensure_pinned_checkout(name: str, destination: Path) -> Path:
    url, commit = PINS[name]
    if not destination.exists():
        destination.parent.mkdir(parents=True, exist_ok=True)
        run(["git", "clone", "--filter=blob:none", url, str(destination)])
    if not (destination / ".git").is_dir():
        raise RuntimeError(f"{destination} exists but is not a git checkout")
    if git_output(destination, "status", "--porcelain", "--untracked-files=no"):
        raise RuntimeError(f"Refusing to change dirty official checkout: {destination}")
    if git_output(destination, "rev-parse", "HEAD") != commit:
        run(["git", "fetch", "origin", commit], cwd=destination)
        run(["git", "checkout", "--detach", commit], cwd=destination)
    actual = git_output(destination, "rev-parse", "HEAD")
    if actual != commit:
        raise RuntimeError(f"Pin mismatch for {name}: expected {commit}, found {actual}")
    return destination


for repo_name, repo_path in REPO_PATHS.items():
    ensure_pinned_checkout(repo_name, repo_path)

print({name: {"path": str(path), "commit": git_output(path, "rev-parse", "HEAD")} for name, path in REPO_PATHS.items()})


+ /media/fezan/ASi/DVLM/steering/aim-flow/.venv/aim-flow/bin/python -m pip install --upgrade pip==24.3.1 setuptools==75.6.0 wheel==0.45.1



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


+ /media/fezan/ASi/DVLM/steering/aim-flow/.venv/aim-flow/bin/python -m pip install torch==2.10.0 torchvision==0.25.0 diffusers==0.30.2 transformers==4.49.0 accelerate==1.13.0 huggingface-hub==0.29.3 safetensors==0.7.0 sentencepiece==0.2.1 protobuf==5.29.3 numpy==2.4.6 Pillow==12.1.1 pandas==3.0.0 tqdm==4.67.1 PyYAML==6.0.3 psutil==7.0.0 nbformat==5.10.4



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


+ /media/fezan/ASi/DVLM/steering/aim-flow/.venv/aim-flow/bin/python -m pip install --no-deps -e /media/fezan/ASi/DVLM/steering/aim-flow
Obtaining file:///media/fezan/ASi/DVLM/steering/aim-flow
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for aim-flow (pyproject.toml): started
  Building editable for aim-flow (pyproject.toml): finished with status 'done'
  Created wheel for aim-flow: filename=aim_flow-0.1.0-0.editable-py3-none-any.whl size=11977 sha256=a9e545095b3bc5dee461f72f0ec12e9011cf559d3a717ad1bd73a2d53bf1bae9
  Stored


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [3]:
# Lock the paper-reproduction protocol, model revision, official code, and hardware.
import gc
import hashlib
import importlib.metadata
import json
import math
import platform
import random
import shutil
import time
import urllib.request
from dataclasses import asdict, dataclass
from typing import Any

import numpy as np
import pandas as pd
import torch
from PIL import Image
from huggingface_hub import HfFolder, model_info, snapshot_download
from tqdm.auto import tqdm

sys.path.insert(0, str(REPO_ROOT / "src"))

@dataclass(frozen=True)
class Protocol:
    model_id: str = "stabilityai/stable-diffusion-3-medium-diffusers"
    model_revision: str = "ea42f8cef0f178587cf766dc8129abd379c90671"
    scheduler: str = "checkpoint FlowMatchEulerDiscreteScheduler"
    dtype: str = "float16"
    full_text_encoders: tuple[str, ...] = ("CLIP-L", "CLIP-G", "T5-XXL")
    max_sequence_length: int = 256
    t5_cache_batch_size: int = 1
    height: int = 1024
    width: int = 1024
    num_inference_steps: int = 28
    cfg_guidance_scale: float = 4.5
    rectified_guidance_scale: float = 7.0
    negative_prompt: str = ""
    subset_prompts: int = 100
    prompts_per_category: int = 25
    selection_seed: int = RUN_SEED
    samples_per_prompt: int = 10
    generation_seed_start: int = RUN_SEED
    rectified_sigma_noise: float = 0.005
    per_device_microbatch: int = 1
    physical_gpu_generation: int = 0
    physical_gpu_evaluator: int = 0

PROTOCOL = Protocol()
METHODS = ("cfg", "rectified_cfgpp")
METHOD_LABELS = {"cfg": "CFG (ω=4.5)", "rectified_cfgpp": "Rectified-CFG++ (λ=7.0)"}
T2I_CATEGORIES = ("color", "texture", "spatial", "shape")
T2I_SEEDS = tuple(PROTOCOL.generation_seed_start + i for i in range(PROTOCOL.samples_per_prompt))
ARTIFACT_ROOT = Path(os.environ.get(
    "AIM_FLOW_RECTCFGPP_REPRO_ROOT",
    REPO_ROOT / "outputs" / f"rectified_cfgpp_paper_reproduction_t2i_100_seed{RUN_SEED}",
)).resolve()
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

if METHODS != ("cfg", "rectified_cfgpp"):
    raise RuntimeError("This notebook is locked to exactly CFG and Rectified-CFG++")
if PROTOCOL.cfg_guidance_scale != 4.5 or PROTOCOL.rectified_guidance_scale != 7.0:
    raise RuntimeError("Expected paper scale pair CFG ω=4.5 and Rectified λ=7.0")
if (PROTOCOL.selection_seed, PROTOCOL.generation_seed_start) != (RUN_SEED, RUN_SEED):
    raise RuntimeError(f"Seed protocol drift: expected {RUN_SEED}")
if PROTOCOL.subset_prompts != 100 or PROTOCOL.prompts_per_category != 25:
    raise RuntimeError("Expected a balanced 100-prompt subset")

def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()

def sha256_file(path: Path, chunk_size: int = 1 << 20) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

def canonical_hash(value: Any) -> str:
    return sha256_bytes(json.dumps(value, sort_keys=True, separators=(",", ":")).encode("utf-8"))

def write_json(value: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    os.replace(temporary, path)

RECTIFIED_PIPELINE_SHA256 = "fd83394f3a9aefbb2b9a706c785aab3dc5251d9b05a8022c5053f37704ef113a"
rectified_pipeline = REPO_PATHS["rectified_cfgpp"] / "rect-cfg-SD3-pipeline" / "pipeline.py"
if sha256_file(rectified_pipeline) != RECTIFIED_PIPELINE_SHA256:
    raise RuntimeError("Pinned official Rectified-CFG++ pipeline hash mismatch")
if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise RuntimeError("This full-T5 reproduction requires two visible CUDA GPUs for conditioning-cache construction")
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN") or HfFolder.get_token()
if not HF_TOKEN:
    raise RuntimeError("Set HF_TOKEN (or log in with huggingface-cli) with accepted SD3 Medium access")
try:
    info = model_info(PROTOCOL.model_id, revision=PROTOCOL.model_revision, token=HF_TOKEN)
    if info.sha != PROTOCOL.model_revision:
        raise RuntimeError(f"Resolved revision {info.sha} != {PROTOCOL.model_revision}")
    MODEL_SNAPSHOT = Path(snapshot_download(
        PROTOCOL.model_id, revision=PROTOCOL.model_revision, token=HF_TOKEN,
    )).resolve()
except Exception as exc:
    raise RuntimeError(f"Cannot access/download the pinned gated SD3 checkpoint: {exc}") from exc
required_model_paths = [
    "transformer", "vae", "text_encoder", "text_encoder_2", "text_encoder_3",
    "tokenizer", "tokenizer_2", "tokenizer_3", "scheduler",
]
missing = [name for name in required_model_paths if not (MODEL_SNAPSHOT / name).exists()]
if missing:
    raise FileNotFoundError(f"Incomplete SD3 snapshot; missing: {missing}")

PROTOCOL_HASH = canonical_hash(asdict(PROTOCOL))
hardware = {
    "platform": platform.platform(), "python": sys.version,
    "torch": torch.__version__, "cuda_runtime": torch.version.cuda,
    "gpus": [
        {"physical_index": i, "name": torch.cuda.get_device_name(i), "memory_bytes": torch.cuda.get_device_properties(i).total_memory}
        for i in range(torch.cuda.device_count())
    ],
}
reproduction_scope = {
    "target": "Rectified-CFG++ NeurIPS 2025 SD3/T2I-CompBench released-code reproduction",
    "paper_scale_pair": {"cfg_omega": 4.5, "rectified_lambda": 7.0, "source": "paper Table 10, SD3 at 28 NFEs"},
    "rectified_implementation": "authors' released custom SD3 pipeline at the pinned official commit",
    "intentional_subset": "100 prompts selected from the four complete official annotation files",
    "disclosure_boundary": "The paper does not identify the exact T2I Table 2 seed or full lambda/mu/sigma record; actual released source is audited.",
}
protocol_record = {
    "protocol": asdict(PROTOCOL), "protocol_hash": PROTOCOL_HASH,
    "model_snapshot": str(MODEL_SNAPSHOT), "hardware": hardware,
    "repository_pins": {name: {"url": PINS[name][0], "commit": PINS[name][1]} for name in ("rectified_cfgpp", "t2i_compbench")},
    "rectified_pipeline_sha256": RECTIFIED_PIPELINE_SHA256,
    "generation_assignment": {f"gpu_{PROTOCOL.physical_gpu_generation}": list(METHODS)},
    "conditioning_topology": "Full CLIP-L + CLIP-G + T5-XXL singleton cache; identical tensors supplied to both methods",
    "reproduction_scope": reproduction_scope,
}
write_json(protocol_record, ARTIFACT_ROOT / "protocol.json")
print(json.dumps(protocol_record, indent=2))


/media/fezan/ASi/DVLM/steering/aim-flow/.venv/aim-flow/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 38 files: 100%|██████████| 38/38 [00:00<00:00, 8842.36it/s]

{
  "protocol": {
    "model_id": "stabilityai/stable-diffusion-3-medium-diffusers",
    "model_revision": "ea42f8cef0f178587cf766dc8129abd379c90671",
    "scheduler": "checkpoint FlowMatchEulerDiscreteScheduler",
    "dtype": "float16",
    "full_text_encoders": [
      "CLIP-L",
      "CLIP-G",
      "T5-XXL"
    ],
    "max_sequence_length": 256,
    "t5_cache_batch_size": 1,
    "height": 1024,
    "width": 1024,
    "num_inference_steps": 28,
    "cfg_guidance_scale": 4.5,
    "rectified_guidance_scale": 7.0,
    "negative_prompt": "",
    "subset_prompts": 100,
    "prompts_per_category": 25,
    "selection_seed": 13,
    "samples_per_prompt": 10,
    "generation_seed_start": 13,
    "rectified_sigma_noise": 0.005,
    "per_device_microbatch": 1,
    "physical_gpu_generation": 0,
    "physical_gpu_evaluator": 0
  },
  "protocol_hash": "71bfc844074344c6643efeca2653c79bb53ce8212fc61b569f8a0e363d138d7b",
  "model_snapshot": "/home/fezan/.cache/huggingface/hub/models--stabilityai--st

In [4]:
# Select 25 prompts per category with the chosen seed, then expand them to ten sequential seeds.
from collections import Counter
from aim_flow.eval_bench.prompt_sources import build_t2i_compbench_manifest

T2I_REPO = REPO_PATHS["t2i_compbench"]
OFFICIAL_DATASET = T2I_REPO / "examples" / "dataset"
SEED25_SELECTED_MANIFEST_CANONICAL_SHA256 = "1d09bc130ee3d752b439511226911a90059cb0833da9176f9820fc82e6534b54"
OFFICIAL_PROMPT_HASHES = {
    "color": "1634259756dbc77d13093d907d414480080ec9790a8c97ad09efaea7b2534f2d",
    "texture": "fbb5363515b4a28009e360afaaaf389d2dd2289e50dbb0c6ef4ea0c7fe4f3d4f",
    "spatial": "8707a1d7e42ce3002d95363cf55f74bd043c843ae27108f5125aaaa7840ca988",
    "shape": "37e1a276906c7ea9516cbd7ac8501be896006c0c17ca7344f260562834455bae",
}
for category, expected_hash in OFFICIAL_PROMPT_HASHES.items():
    source = OFFICIAL_DATASET / f"{category}_val.txt"
    if sha256_file(source) != expected_hash:
        raise RuntimeError(f"Pinned official prompt file hash mismatch: {source}")

rebuilt = build_t2i_compbench_manifest(
    subset_size=PROTOCOL.subset_prompts,
    seed=PROTOCOL.selection_seed,
    dataset_root=OFFICIAL_DATASET,
)
selected_manifest_hash = canonical_hash(rebuilt.to_dict())
if PROTOCOL.selection_seed == 25 and selected_manifest_hash != SEED25_SELECTED_MANIFEST_CANONICAL_SHA256:
    raise RuntimeError("Fresh random.Random(25) selection differs from the verified seed-25 baseline")
counts = Counter(sample.category for sample in rebuilt.samples)
if counts != Counter({category: PROTOCOL.prompts_per_category for category in T2I_CATEGORIES}):
    raise RuntimeError(f"Subset is not balanced {PROTOCOL.prompts_per_category}/category: {counts}")
for sample in rebuilt.samples:
    if "_" in sample.prompt or "/" in sample.prompt or "\\" in sample.prompt:
        raise ValueError(f"Prompt is incompatible with the official scorer filename parser: {sample.prompt!r}")

tasks = []
for sample in rebuilt.samples:
    category_rank = int(sample.id.rsplit("_", 1)[1])
    for sample_index, seed in enumerate(T2I_SEEDS):
        tasks.append({
            "subset_sample_id": sample.id,
            "category": sample.category,
            "prompt": sample.prompt,
            "official_prompt_index": int(sample.metadata["original_index"]),
            "selected_category_rank": category_rank,
            "sample_index": sample_index,
            "question_id": category_rank * PROTOCOL.samples_per_prompt + sample_index,
            "seed": seed,
        })
if len(tasks) != 1000 or len({(task["category"], task["question_id"]) for task in tasks}) != 1000:
    raise RuntimeError("Expected exactly 1,000 unique generation tasks")

MANIFEST_DIR = ARTIFACT_ROOT / "manifests"
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
SELECTED_MANIFEST = MANIFEST_DIR / "selected_prompts.json"
rebuilt.save(SELECTED_MANIFEST)
TASK_MANIFEST = MANIFEST_DIR / "generation_tasks.jsonl"
TASK_MANIFEST.write_text("\n".join(json.dumps(task, sort_keys=True) for task in tasks) + "\n", encoding="utf-8")
for category in T2I_CATEGORIES:
    selected = [sample.prompt for sample in rebuilt.samples if sample.category == category]
    (MANIFEST_DIR / f"{category}_selected_seed{PROTOCOL.selection_seed}.txt").write_text("\n".join(selected) + "\n", encoding="utf-8")
subset_audit = {
    "algorithm": f"random.Random({PROTOCOL.selection_seed}).sample independently per category; indices sorted into official order",
    "selection_seed": PROTOCOL.selection_seed,
    "counts": dict(counts), "task_count": len(tasks),
    "generation_seeds": list(T2I_SEEDS),
    "official_question_id_rule": "category-local prompt rank * 10 + sample index (000000..000249)",
    "selected_manifest_canonical_sha256": selected_manifest_hash,
    "seed25_baseline_verified": PROTOCOL.selection_seed != 25 or selected_manifest_hash == SEED25_SELECTED_MANIFEST_CANONICAL_SHA256,
    "selected_manifest_file_sha256": sha256_file(SELECTED_MANIFEST),
    "task_manifest_sha256": sha256_file(TASK_MANIFEST),
    "source_prompt_hashes": OFFICIAL_PROMPT_HASHES,
}
write_json(subset_audit, MANIFEST_DIR / "subset_audit.json")
print(json.dumps(subset_audit, indent=2))


{
  "algorithm": "random.Random(13).sample independently per category; indices sorted into official order",
  "selection_seed": 13,
  "counts": {
    "color": 25,
    "shape": 25,
    "texture": 25,
    "spatial": 25
  },
  "task_count": 1000,
  "generation_seeds": [
    13,
    14,
    15,
    16,
    17,
    18,
    19,
    20,
    21,
    22
  ],
  "official_question_id_rule": "category-local prompt rank * 10 + sample index (000000..000249)",
  "selected_manifest_canonical_sha256": "3fcc091d66f2e3d673a4e106aa27b15518262ab9db848ef36d10740e8f699366",
  "seed25_baseline_verified": true,
  "selected_manifest_file_sha256": "f9cf97555168f7ef825e841b8db717c4c7b12e4ef36a1aa75678bd10445cdf9e",
  "task_manifest_sha256": "b89b63dcadc134989a96207de326c8409820f955539b5d8c06074077591b04c5",
  "source_prompt_hashes": {
    "color": "1634259756dbc77d13093d907d414480080ec9790a8c97ad09efaea7b2534f2d",
    "texture": "fbb5363515b4a28009e360afaaaf389d2dd2289e50dbb0c6ef4ea0c7fe4f3d4f",
    "spatial": "8

In [5]:
# Encode every text once with the full, unmodified SD3 text stack. T5-XXL is sharded over both GPUs.
from safetensors import safe_open
from safetensors.torch import load_file as load_safetensors, save_file as save_safetensors

if PROTOCOL.t5_cache_batch_size != 1:
    raise RuntimeError("Stale notebook state detected: rerun the protocol cell so singleton T5 caching is active.")

EMBEDDING_ROOT = ARTIFACT_ROOT / "full_t5_prompt_cache"
EMBEDDING_ROOT.mkdir(parents=True, exist_ok=True)
EMBEDDING_INDEX = EMBEDDING_ROOT / "index.json"

CACHE_IDENTITY = {
    "schema": "sd3_full_text_condition_v1",
    "model_id": PROTOCOL.model_id,
    "model_revision": PROTOCOL.model_revision,
    "diffusers": importlib.metadata.version("diffusers"),
    "transformers": importlib.metadata.version("transformers"),
    "dtype": PROTOCOL.dtype,
    "max_sequence_length": PROTOCOL.max_sequence_length,
    "clip_skip": None,
    "prompt_routing": "same exact text to CLIP-L, CLIP-G, and T5-XXL",
    "full_t5": True,
    "encoding_batch_size": PROTOCOL.t5_cache_batch_size,
    "encoding_topology": "singleton_per_exact_text",
}
CACHE_IDENTITY_HASH = canonical_hash(CACHE_IDENTITY)


def embedding_key(text: str) -> str:
    return canonical_hash({"cache_identity": CACHE_IDENTITY, "text": text})


def embedding_path(text: str) -> Path:
    return EMBEDDING_ROOT / f"{embedding_key(text)}.safetensors"


required_texts = sorted({PROTOCOL.negative_prompt, *(sample.prompt for sample in rebuilt.samples)})


def valid_embedding(text: str) -> bool:
    path = embedding_path(text)
    try:
        with safe_open(str(path), framework="pt", device="cpu") as handle:
            if set(handle.keys()) != {"pooled_prompt_embeds", "prompt_embeds"}:
                return False
            prompt_shape = tuple(handle.get_slice("prompt_embeds").get_shape())
            pooled_shape = tuple(handle.get_slice("pooled_prompt_embeds").get_shape())
            metadata = handle.metadata()
        return (
            prompt_shape == (1, 333, 4096)
            and pooled_shape == (1, 2048)
            and metadata.get("cache_identity_sha256") == CACHE_IDENTITY_HASH
            and metadata.get("entry_key") == embedding_key(text)
        )
    except Exception:
        return False


# A failed prior audit can leave the full pipeline referenced in the live kernel.
# Release only these cell-owned objects before validating/resuming the cache.
def release_full_text_stack():
    for stale_name in ("normal_cfg", "cached_positive", "cached_negative", "full_pipe", "tokenizer_3", "t5"):
        stale_object = globals().pop(stale_name, None)
        del stale_object
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()


release_full_text_stack()

previous_index = None
if EMBEDDING_INDEX.exists():
    try:
        previous_index = json.loads(EMBEDDING_INDEX.read_text(encoding="utf-8"))
        if previous_index.get("cache_identity_sha256") != CACHE_IDENTITY_HASH:
            previous_index = None
    except Exception:
        previous_index = None
pending = [text for text in required_texts if not valid_embedding(text)]
print(f"Full-T5 cache: {len(required_texts) - len(pending)}/{len(required_texts)} valid; {len(pending)} pending")
equivalence_audit = None if previous_index is None else previous_index.get("singleton_cfg_exact_equivalence_audit")
t5_device_map = None if previous_index is None else previous_index.get("t5_device_map")
needs_audit = not equivalence_audit or not equivalence_audit.get("passed", False)
if pending or needs_audit:
    # Jupyter fires post_run_cell after both successful and failed execution.
    # This one-shot guard releases a partially constructed sharded T5 stack on any exception.
    def _release_full_text_stack_after_cell(result):
        try:
            release_full_text_stack()
        finally:
            try:
                get_ipython().events.unregister("post_run_cell", _release_full_text_stack_after_cell)
            except ValueError:
                pass

    get_ipython().events.register("post_run_cell", _release_full_text_stack_after_cell)
    from diffusers import StableDiffusion3Pipeline
    from transformers import T5EncoderModel, T5TokenizerFast

    total_gib = [torch.cuda.get_device_properties(i).total_memory // (1024 ** 3) for i in range(2)]
    max_memory = {
        0: f"{max(4, int(total_gib[0]) - 2)}GiB",
        1: f"{max(4, int(total_gib[1]) - 2)}GiB",
        "cpu": "45GiB",
    }
    t5 = T5EncoderModel.from_pretrained(
        MODEL_SNAPSHOT / "text_encoder_3", torch_dtype=torch.float16,
        low_cpu_mem_usage=True, device_map="balanced", max_memory=max_memory,
    )
    t5_device_map = dict(t5.hf_device_map)
    tokenizer_3 = T5TokenizerFast.from_pretrained(MODEL_SNAPSHOT / "tokenizer_3")
    full_pipe = StableDiffusion3Pipeline.from_pretrained(
        str(MODEL_SNAPSHOT), torch_dtype=torch.float16, use_safetensors=True,
        low_cpu_mem_usage=True, text_encoder_3=t5, tokenizer_3=tokenizer_3,
    )
    full_pipe.set_progress_bar_config(disable=True)

    encode_common = dict(
        prompt_2=None, prompt_3=None, negative_prompt=None,
        negative_prompt_2=None, negative_prompt_3=None,
        do_classifier_free_guidance=False, device=torch.device("cpu"),
        num_images_per_prompt=1, max_sequence_length=PROTOCOL.max_sequence_length,
    )
    for start in tqdm(range(0, len(pending), PROTOCOL.t5_cache_batch_size), desc="Full SD3/T5 prompt encoding"):
        batch = pending[start:start + PROTOCOL.t5_cache_batch_size]
        with torch.inference_mode():
            encoded = full_pipe.encode_prompt(prompt=batch, **encode_common)
        prompt_batch, pooled_batch = encoded[0].cpu(), encoded[2].cpu()
        if prompt_batch.shape[1:] != (333, 4096) or pooled_batch.shape[1:] != (2048,):
            raise RuntimeError(f"Unexpected full-SD3 embedding shapes: {prompt_batch.shape}, {pooled_batch.shape}")
        for index, text_value in enumerate(batch):
            destination = embedding_path(text_value)
            temporary = destination.with_suffix(".part.safetensors")
            save_safetensors({
                "prompt_embeds": prompt_batch[index:index + 1].contiguous(),
                "pooled_prompt_embeds": pooled_batch[index:index + 1].contiguous(),
            }, str(temporary), metadata={
                "entry_key": embedding_key(text_value),
                "text_sha256": sha256_bytes(text_value.encode("utf-8")),
                "cache_identity_sha256": CACHE_IDENTITY_HASH,
                "model_revision": PROTOCOL.model_revision,
                "max_sequence_length": str(PROTOCOL.max_sequence_length),
                "text_encoders": ",".join(PROTOCOL.full_text_encoders),
            })
            roundtrip = load_safetensors(str(temporary), device="cpu")
            if not (
                torch.equal(roundtrip["prompt_embeds"], prompt_batch[index:index + 1])
                and torch.equal(roundtrip["pooled_prompt_embeds"], pooled_batch[index:index + 1])
            ):
                raise RuntimeError(f"Safetensors round-trip changed cached conditioning: {text_value!r}")
            del roundtrip
            os.replace(temporary, destination)
        del encoded, prompt_batch, pooled_batch

    # The cache uses singleton encoding, exactly matching per-prompt CFG topology.
    # Require bitwise equality for positive, negative, and pooled tensors.
    audit_prompt = rebuilt.samples[0].prompt
    with torch.inference_mode():
        normal_cfg = full_pipe.encode_prompt(
            prompt=audit_prompt, prompt_2=None, prompt_3=None,
            negative_prompt=PROTOCOL.negative_prompt, negative_prompt_2=None, negative_prompt_3=None,
            do_classifier_free_guidance=True, device=torch.device("cpu"),
            num_images_per_prompt=1, max_sequence_length=PROTOCOL.max_sequence_length,
        )
    cached_positive = load_safetensors(str(embedding_path(audit_prompt)), device="cpu")
    cached_negative = load_safetensors(str(embedding_path(PROTOCOL.negative_prompt)), device="cpu")
    def tensor_difference_stats(reference, candidate):
        reference_f32 = reference.float()
        candidate_f32 = candidate.float()
        difference = (reference_f32 - candidate_f32).abs()
        reference_norm = max(float(reference_f32.norm()), torch.finfo(torch.float32).tiny)
        reference_abs_max = max(float(reference_f32.abs().max()), torch.finfo(torch.float32).tiny)
        return {
            "exact": bool(torch.equal(reference, candidate)),
            "finite": bool(torch.isfinite(candidate).all()),
            "max_abs": float(difference.max()),
            "mean_abs": float(difference.mean()),
            "rmse": float(difference.square().mean().sqrt()),
            "relative_l2": float((reference_f32 - candidate_f32).norm()) / reference_norm,
            "max_abs_over_reference_max": float(difference.max()) / reference_abs_max,
        }

    positive_prompt_stats = tensor_difference_stats(normal_cfg[0], cached_positive["prompt_embeds"])
    positive_pooled_stats = tensor_difference_stats(normal_cfg[2], cached_positive["pooled_prompt_embeds"])
    negative_prompt_stats = tensor_difference_stats(normal_cfg[1], cached_negative["prompt_embeds"])
    negative_pooled_stats = tensor_difference_stats(normal_cfg[3], cached_negative["pooled_prompt_embeds"])
    audit_passed = (
        positive_prompt_stats["exact"] and negative_prompt_stats["exact"]
        and positive_pooled_stats["exact"] and negative_pooled_stats["exact"]
    )
    equivalence_audit = {
        "prompt": audit_prompt,
        "comparison": "cached singleton FP16 encode versus singleton FP16 CFG encode",
        "positive_prompt": positive_prompt_stats,
        "positive_pooled": positive_pooled_stats,
        "negative_prompt": negative_prompt_stats,
        "negative_pooled": negative_pooled_stats,
        "passed": bool(audit_passed),
    }
    audit_failure = None if audit_passed else RuntimeError(
        f"Cached singleton conditioning is not bitwise-identical to singleton CFG encoding: {equivalence_audit}"
    )
    release_full_text_stack()
    if audit_failure is not None:
        raise audit_failure

invalid = [text for text in required_texts if not valid_embedding(text)]
if invalid:
    raise RuntimeError(f"Full-T5 cache remains incomplete: {len(invalid)} invalid entries")
index_record = {
    "protocol_hash": PROTOCOL_HASH,
    "model_id": PROTOCOL.model_id, "model_revision": PROTOCOL.model_revision,
    "cache_identity": CACHE_IDENTITY,
    "cache_identity_sha256": CACHE_IDENTITY_HASH,
    "full_text_encoders": list(PROTOCOL.full_text_encoders),
    "max_sequence_length": PROTOCOL.max_sequence_length,
    "entry_count": len(required_texts),
    "entries": {
        text: {
            "key": embedding_key(text),
            "file": embedding_path(text).name,
            "sha256": sha256_file(embedding_path(text)),
            "prompt_shape": [1, 333, 4096],
            "pooled_shape": [1, 2048],
            "dtype": "torch.float16",
        }
        for text in required_texts
    },
    "t5_device_map": t5_device_map,
    "singleton_cfg_exact_equivalence_audit": equivalence_audit,
}
write_json(index_record, EMBEDDING_INDEX)
print(f"Validated {len(required_texts)} full-SD3/T5 conditioning entries ({EMBEDDING_INDEX})")


Full-T5 cache: 101/101 valid; 0 pending
Validated 101 full-SD3/T5 conditioning entries (/media/fezan/ASi/DVLM/steering/aim-flow/outputs/rectified_cfgpp_paper_reproduction_t2i_100_seed13/full_t5_prompt_cache/index.json)


In [6]:
# Verify the dedicated two-method worker and audit the exact authors' pipeline that executes.
WORKER_PATH = REPO_ROOT / "scripts" / "reproduce_rectified_cfgpp_t2i_worker.py"
EXPECTED_WORKER_SHA256 = "ee8032140a638b74dd2534ceb8c0b8ef4c0708ed7106206bceb2cb54bbceaddb"
if sha256_file(WORKER_PATH) != EXPECTED_WORKER_SHA256:
    raise RuntimeError("Reproduction worker changed; update the locked hash only after deliberate review")
compile(WORKER_PATH.read_text(encoding="utf-8"), str(WORKER_PATH), "exec")
released_source = rectified_pipeline.read_text(encoding="utf-8")
released_pipeline_algorithm_audit = {
    "pipeline_sha256": sha256_file(rectified_pipeline),
    "has_true_cfg_parameter": "true_cfg: float = 4.5" in released_source,
    "has_sigma_noise_default": "sigma_noise: float = 0.005" in released_source,
    "adaptive_lambda_schedule_enabled": "current_lambda = lambda_schedule[i].item()" in released_source.replace("# current_lambda", "DISABLED"),
    "released_predictor_expression": "x_pred = latents + dt * noise_pred" in released_source,
    "released_corrector_time_expression": "timestep = (t+dt).expand" in released_source,
    "released_guidance_expression": "noise_pred = noise_pred_text + true_cfg * (noise_pred_text_2 - noise_pred_uncond_2)" in released_source,
    "interpretation": "Audit of the authors' executable release; paper Algorithm 1 and released source are not textually identical.",
}
if not all(released_pipeline_algorithm_audit[key] for key in (
    "has_true_cfg_parameter", "has_sigma_noise_default",
    "released_predictor_expression", "released_guidance_expression",
)):
    raise RuntimeError("Pinned released pipeline no longer matches the audited implementation")
write_json(released_pipeline_algorithm_audit, ARTIFACT_ROOT / "released_pipeline_algorithm_audit.json")
print(json.dumps(released_pipeline_algorithm_audit, indent=2))


{
  "pipeline_sha256": "fd83394f3a9aefbb2b9a706c785aab3dc5251d9b05a8022c5053f37704ef113a",
  "has_true_cfg_parameter": true,
  "has_sigma_noise_default": true,
  "adaptive_lambda_schedule_enabled": false,
  "released_predictor_expression": true,
  "released_corrector_time_expression": true,
  "released_guidance_expression": true,
  "interpretation": "Audit of the authors' executable release; paper Algorithm 1 and released source are not textually identical."
}


In [7]:
# Launch both methods sequentially on the same GPU. Rerunning resumes checksum-valid pairs.
import subprocess
ASSIGNMENTS = {PROTOCOL.physical_gpu_generation: list(METHODS)}
WORKER_LOG_DIR = ARTIFACT_ROOT / "logs" / "generation"
WORKER_LOG_DIR.mkdir(parents=True, exist_ok=True)
processes, log_handles = {}, {}
for physical_gpu, assigned_methods in ASSIGNMENTS.items():
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(physical_gpu)
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONHASHSEED"] = str(PROTOCOL.generation_seed_start)
    env["PYTHONPATH"] = str(REPO_ROOT / "src") + os.pathsep + env.get("PYTHONPATH", "")
    command = [
        sys.executable, str(WORKER_PATH),
        "--repo-root", str(REPO_ROOT),
        "--artifact-root", str(ARTIFACT_ROOT),
        "--model-snapshot", str(MODEL_SNAPSHOT),
        "--rectified-repo", str(REPO_PATHS["rectified_cfgpp"]),
        "--protocol", str(ARTIFACT_ROOT / "protocol.json"),
        "--tasks", str(TASK_MANIFEST),
        "--embedding-index", str(EMBEDDING_INDEX),
        "--physical-gpu", str(physical_gpu),
        "--methods", *assigned_methods,
    ]
    log_path = WORKER_LOG_DIR / f"gpu_{physical_gpu}.log"
    handle = log_path.open("a", encoding="utf-8")
    handle.write("\n+ " + " ".join(command) + "\n")
    handle.flush()
    processes[physical_gpu] = subprocess.Popen(command, env=env, stdout=handle, stderr=subprocess.STDOUT, text=True)
    log_handles[physical_gpu] = handle
    print(f"GPU {physical_gpu}: {assigned_methods}; PID={processes[physical_gpu].pid}; log={log_path}")

def generation_counts():
    return {
        method: sum(1 for _ in (ARTIFACT_ROOT / "t2i_compbench" / method).glob("*/samples/*.png"))
        for method in METHODS
    }

try:
    while any(process.poll() is None for process in processes.values()):
        print(time.strftime("%Y-%m-%d %H:%M:%S"), generation_counts(), {gpu: p.poll() for gpu, p in processes.items()})
        time.sleep(30)
except KeyboardInterrupt:
    for process in processes.values():
        if process.poll() is None:
            process.terminate()
    raise
finally:
    for handle in log_handles.values():
        handle.close()
failures = {gpu: process.returncode for gpu, process in processes.items() if process.returncode != 0}
if failures:
    raise RuntimeError(f"Generation worker failure(s): {failures}. Inspect {WORKER_LOG_DIR}")
print("Generation workers completed:", generation_counts())


GPU 0: ['cfg', 'rectified_cfgpp']; PID=445288; log=/media/fezan/ASi/DVLM/steering/aim-flow/outputs/rectified_cfgpp_paper_reproduction_t2i_100_seed13/logs/generation/gpu_0.log
2026-08-12 18:56:11 {'cfg': 384, 'rectified_cfgpp': 0} {0: None}
2026-08-12 18:56:41 {'cfg': 385, 'rectified_cfgpp': 0} {0: None}
2026-08-12 18:57:11 {'cfg': 387, 'rectified_cfgpp': 0} {0: None}
2026-08-12 18:57:41 {'cfg': 389, 'rectified_cfgpp': 0} {0: None}
2026-08-12 18:58:11 {'cfg': 390, 'rectified_cfgpp': 0} {0: None}
2026-08-12 18:58:41 {'cfg': 392, 'rectified_cfgpp': 0} {0: None}
2026-08-12 18:59:11 {'cfg': 394, 'rectified_cfgpp': 0} {0: None}
2026-08-12 18:59:41 {'cfg': 396, 'rectified_cfgpp': 0} {0: None}
2026-08-12 19:00:11 {'cfg': 398, 'rectified_cfgpp': 0} {0: None}
2026-08-12 19:00:41 {'cfg': 399, 'rectified_cfgpp': 0} {0: None}
2026-08-12 19:01:11 {'cfg': 401, 'rectified_cfgpp': 0} {0: None}
2026-08-12 19:01:41 {'cfg': 403, 'rectified_cfgpp': 0} {0: None}
2026-08-12 19:02:11 {'cfg': 405, 'rectified_c

In [8]:
# Validate counts, checksums, prompt/seed pairing, guidance parameters, commit, and scheduler parity.
generation_audit = {"protocol_hash": PROTOCOL_HASH, "methods": {}}
scheduler_fingerprints, scheduler_classes = {}, {}
task_keys = {(task["category"], task["question_id"]): task for task in tasks}
for method in METHODS:
    method_record, seen = {"categories": {}}, set()
    expected_scale = PROTOCOL.cfg_guidance_scale if method == "cfg" else PROTOCOL.rectified_guidance_scale
    for category in T2I_CATEGORIES:
        stage = ARTIFACT_ROOT / "t2i_compbench" / method / category
        images = sorted((stage / "samples").glob("*.png"))
        sidecars = sorted((stage / "sample_metadata").glob("*.json"))
        if len(images) != 250 or len(sidecars) != 250:
            raise RuntimeError(f"Expected 250 pairs for {method}/{category}; got {len(images)}/{len(sidecars)}")
        image_by_qid = {int(path.stem.rsplit("_", 1)[1]): path for path in images}
        metadata_by_qid = {int(path.stem): path for path in sidecars}
        if set(image_by_qid) != set(range(250)) or set(image_by_qid) != set(metadata_by_qid):
            raise RuntimeError(f"Question IDs are not exactly 000000..000249 for {method}/{category}")
        hash_rows = []
        for question_id, image_path in image_by_qid.items():
            task = task_keys[(category, question_id)]
            metadata = json.loads(metadata_by_qid[question_id].read_text(encoding="utf-8"))
            checks = {
                "prompt": task["prompt"], "seed": task["seed"], "sample_index": task["sample_index"],
                "protocol_hash": PROTOCOL_HASH, "method": method, "full_t5_conditioning": True,
                "guidance_scale": expected_scale,
                "guidance_parameter_name": "omega" if method == "cfg" else "lambda",
            }
            if any(metadata.get(key) != value for key, value in checks.items()):
                raise RuntimeError(f"Sidecar mismatch: {metadata_by_qid[question_id]}")
            if method == "rectified_cfgpp" and metadata.get("rectified_cfgpp_commit") != PINS["rectified_cfgpp"][1]:
                raise RuntimeError(f"Rectified commit mismatch: {metadata_by_qid[question_id]}")
            if metadata["image_sha256"] != sha256_file(image_path):
                raise RuntimeError(f"Image checksum mismatch: {image_path}")
            seen.add((category, question_id))
            hash_rows.append((image_path.name, metadata["image_sha256"]))
        method_record["categories"][category] = {"count": len(images), "image_set_sha256": canonical_hash(sorted(hash_rows))}
    if seen != set(task_keys):
        raise RuntimeError(f"Task coverage mismatch: {method}")
    run_record = json.loads((ARTIFACT_ROOT / "t2i_compbench" / method / "generation_run.json").read_text())
    if float(run_record["guidance_scale"]) != float(expected_scale):
        raise RuntimeError(f"Run guidance mismatch: {method}")
    scheduler_fingerprints[method] = canonical_hash(run_record["scheduler"]["semantic_config"])
    scheduler_classes[method] = run_record["scheduler"]["class"]
    method_record.update(guidance_scale=expected_scale, scheduler=run_record["scheduler"], total=len(seen))
    generation_audit["methods"][method] = method_record
if len(set(scheduler_fingerprints.values())) != 1 or len(set(scheduler_classes.values())) != 1:
    raise RuntimeError(f"Schedulers differ: {scheduler_fingerprints}, {scheduler_classes}")
generation_audit["shared_scheduler_fingerprint"] = next(iter(scheduler_fingerprints.values()))
write_json(generation_audit, ARTIFACT_ROOT / "t2i_compbench" / "generation_audit.json")
print(json.dumps({method: generation_audit["methods"][method]["total"] for method in METHODS}, indent=2))


{
  "cfg": 1000,
  "rectified_cfgpp": 1000
}


## Official T2I-CompBench scoring

The next cells execute the pinned repository's official metric code without reimplementation: BLIP-VQA for color, texture, and shape; UniDet RS200 for 2D spatial relationships. Each category has 25 prompts × 10 images = 250 inputs per method.

**Documented execution adaptations:** Detectron2 imports PyTorch while pip prepares metadata, so the same official PyTorch/Torchvision pins are installed first, followed by the remaining lock and exact pinned Detectron2 commit. The upstream 2D spatial script also hard-codes a batch of 64, which exhausts this 11 GB GPU for this 1024² subset. A hash-locked launcher changes only that runtime batch to the largest verified safe value, 32; the pinned source, model, weights, preprocessing, detections, and scoring logic remain unchanged. The strict probe checks CUDA, the compiled extension, spaCy, versions, and commit before scoring.


In [9]:
# Create/reuse the pinned isolated official evaluator environment and verify the UniDet checkpoint.
T2I_ENV = REPO_ROOT / ".venv" / "t2i-compbench-py310"
T2I_ENV_PYTHON = T2I_ENV / "bin" / "python"
T2I_LOCK_DIR = ARTIFACT_ROOT / "environment_locks"
T2I_LOCK_DIR.mkdir(parents=True, exist_ok=True)
T2I_REQUIREMENTS_LOCK = T2I_LOCK_DIR / "t2i_compbench_requirements_pinned.txt"
T2I_STAGED_REQUIREMENTS_LOCK = T2I_LOCK_DIR / "t2i_compbench_requirements_staged.txt"
T2I_TORCH_BOOTSTRAP = ("torch==2.0.1", "torchvision==0.15.2")
T2I_DETECTRON2_COMMIT = PINS.get("detectron2", (None, "5aeb252b194b93dc2879b4ac34bc51a31b5aee13"))[1]
T2I_ENV_BUILDER_SCHEMA = "official_t2i_cuda117_staged_v2"


def conda_executable() -> str:
    executable = shutil.which("conda")
    if not executable:
        raise FileNotFoundError("conda is required for the official T2I-CompBench evaluator environment")
    return executable


def ensure_t2i_environment():
    if not T2I_ENV_PYTHON.exists():
        run([conda_executable(), "create", "-y", "-p", str(T2I_ENV), "python=3.10", "pip=23.2.1"])
    build_env = os.environ.copy()
    build_env["CUDA_HOME"] = str(T2I_ENV)
    build_env["PATH"] = str(T2I_ENV / "bin") + os.pathsep + build_env.get("PATH", "")
    build_env["LD_LIBRARY_PATH"] = str(T2I_ENV / "lib") + os.pathsep + build_env.get("LD_LIBRARY_PATH", "")
    build_env["CC"] = str(T2I_ENV / "bin" / "x86_64-conda-linux-gnu-gcc")
    build_env["CXX"] = str(T2I_ENV / "bin" / "x86_64-conda-linux-gnu-g++")
    build_env["CPATH"] = str(T2I_ENV / "include") + os.pathsep + build_env.get("CPATH", "")
    build_env["CFLAGS"] = f"-I{T2I_ENV / 'include'}"
    build_env["CXXFLAGS"] = f"-I{T2I_ENV / 'include'}"
    build_env["TORCH_CUDA_ARCH_LIST"] = ";".join(sorted({
        f"{major}.{minor}" for major, minor in
        (torch.cuda.get_device_capability(i) for i in range(torch.cuda.device_count()))
    }))
    build_env["FORCE_CUDA"] = "1"
    build_env["MAX_JOBS"] = "2"
    official = (T2I_REPO / "requirements.txt").read_text(encoding="utf-8")
    official = official.replace(
        "git+https://github.com/openai/CLIP.git",
        f"git+https://github.com/openai/CLIP.git@{PINS['openai_clip'][1]}",
    )
    T2I_REQUIREMENTS_LOCK.write_text(official, encoding="utf-8")
    staged_lines = [
        line for line in official.splitlines()
        if not line.startswith("git+https://github.com/facebookresearch/detectron2.git")
        and not line.startswith("torch==")
        and not line.startswith("torchvision==")
    ]
    T2I_STAGED_REQUIREMENTS_LOCK.write_text("\n".join(staged_lines) + "\n", encoding="utf-8")
    marker = T2I_ENV / ".requirements_sha256"
    lock_hash = canonical_hash({
        "official_requirements_sha256": sha256_file(T2I_REQUIREMENTS_LOCK),
        "builder_schema": T2I_ENV_BUILDER_SCHEMA,
        "detectron2_commit": T2I_DETECTRON2_COMMIT,
        "torch_bootstrap": T2I_TORCH_BOOTSTRAP,
    })
    probe_code = (
        "import importlib.metadata as md, json, torch, torchvision, transformers, detectron2, detectron2._C, spacy; "
        "assert torch.cuda.is_available(); assert torch.__version__.startswith('2.0.1'); "
        "assert torchvision.__version__.startswith('0.15.2'); assert transformers.__version__ == '4.30.2'; "
        "assert detectron2._C.has_cuda(); spacy.load('en_core_web_sm'); "
        "d=json.loads(md.distribution('detectron2').read_text('direct_url.json')); "
        f"assert d['vcs_info']['commit_id'] == '{T2I_DETECTRON2_COMMIT}'; "
        "print(torch.__version__, torch.version.cuda, transformers.__version__, "
        "detectron2._C.get_cuda_version(), d['vcs_info']['commit_id'])"
    )
    probe = subprocess.run(
        [str(T2I_ENV_PYTHON), "-c", probe_code], cwd=str(REPO_ROOT), env=build_env,
        text=True, capture_output=True,
    )
    if probe.returncode != 0:
        # Detectron2 imports torch while preparing its package metadata; install
        # the official CUDA 11.7 torch pins before any source build.
        run([str(T2I_ENV_PYTHON), "-m", "pip", "install", "--no-cache-dir",
             *T2I_TORCH_BOOTSTRAP, "--index-url", "https://download.pytorch.org/whl/cu117"])
        run([str(T2I_ENV_PYTHON), "-m", "pip", "install", "setuptools<70", "ninja==1.13.0"])
        run([conda_executable(), "install", "-y", "-p", str(T2I_ENV),
             "gcc_linux-64=11", "gxx_linux-64=11"])
        run([conda_executable(), "install", "-y", "-p", str(T2I_ENV),
             "-c", "nvidia/label/cuda-11.7.1", "-c", "nvidia",
             "cuda-nvcc=11.7.99", "cuda-cccl=11.7.91",
             "cuda-cudart=11.7.99", "cuda-cudart-dev=11.7.99",
             "cuda-driver-dev=11.7.99", "cuda-nvrtc=11.7.99", "cuda-nvrtc-dev=11.7.99",
             "libcublas=11.10.3.66", "libcublas-dev=11.10.3.66",
             "libcusparse=11.7.4.91", "libcusparse-dev=11.7.4.91",
             "libcusolver=11.4.0.1", "libcusolver-dev=11.4.0.1", "cuda-version=11.7"])
        run([str(T2I_ENV_PYTHON), "-m", "pip", "install", "--no-cache-dir",
             "-r", str(T2I_STAGED_REQUIREMENTS_LOCK)], env=build_env)
        run([str(T2I_ENV_PYTHON), "-m", "pip", "install", "--no-cache-dir",
             "--no-build-isolation", "--no-deps",
             f"git+https://github.com/facebookresearch/detectron2.git@{T2I_DETECTRON2_COMMIT}"], env=build_env)
    else:
        print("Pinned CUDA evaluator environment already passes the strict probe; skipping rebuild.")
    check = run([str(T2I_ENV_PYTHON), "-c", probe_code], env=build_env, capture=True)
    print(check.stdout.strip())
    if not marker.exists() or marker.read_text().strip() != lock_hash:
        marker.write_text(lock_hash + "\n", encoding="utf-8")
    frozen = run([str(T2I_ENV_PYTHON), "-m", "pip", "freeze", "--all"], capture=True)
    (T2I_LOCK_DIR / "t2i_compbench_pip_freeze.txt").write_text(frozen.stdout, encoding="utf-8")


ensure_t2i_environment()

UNIDET_WEIGHT = T2I_REPO / "UniDet_eval" / "experts" / "expert_weights" / "Unified_learned_OCIM_RS200_6x+2x.pth"
UNIDET_URL = "https://huggingface.co/shikunl/prismer/resolve/main/expert_weights/Unified_learned_OCIM_RS200_6x%2B2x.pth"
UNIDET_SHA256 = "b19a836811c0d0d7aebf04f83ff4ea1da77ccf57b85cd65819746c1fccbff258"


def download_verified(url: str, destination: Path, expected_sha256: str):
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.exists():
        temporary = destination.with_suffix(destination.suffix + ".part")
        urllib.request.urlretrieve(url, temporary)
        os.replace(temporary, destination)
    actual = sha256_file(destination)
    if actual != expected_sha256:
        raise RuntimeError(f"Checkpoint checksum mismatch for {destination}: {actual}")
    return destination


download_verified(UNIDET_URL, UNIDET_WEIGHT, UNIDET_SHA256)
print("Official scorer environment and UniDet checkpoint: READY")


Pinned CUDA evaluator environment already passes the strict probe; skipping rebuild.
+ /media/fezan/ASi/DVLM/steering/aim-flow/.venv/t2i-compbench-py310/bin/python -c import importlib.metadata as md, json, torch, torchvision, transformers, detectron2, detectron2._C, spacy; assert torch.cuda.is_available(); assert torch.__version__.startswith('2.0.1'); assert torchvision.__version__.startswith('0.15.2'); assert transformers.__version__ == '4.30.2'; assert detectron2._C.has_cuda(); spacy.load('en_core_web_sm'); d=json.loads(md.distribution('detectron2').read_text('direct_url.json')); assert d['vcs_info']['commit_id'] == '5aeb252b194b93dc2879b4ac34bc51a31b5aee13'; print(torch.__version__, torch.version.cuda, transformers.__version__, detectron2._C.get_cuda_version(), d['vcs_info']['commit_id'])
2.0.1+cu117 11.7 4.30.2 CUDA 11.7 5aeb252b194b93dc2879b4ac34bc51a31b5aee13
+ /media/fezan/ASi/DVLM/steering/aim-flow/.venv/t2i-compbench-py310/bin/python -m pip freeze --all
Official scorer environ

In [10]:
# Run the official scorers, retain raw outputs/logs, and aggregate the four subset scores.
UNIDET_RUNNER = REPO_ROOT / "scripts" / "run_t2i_unidet_spatial_memory_safe.py"
EXPECTED_UNIDET_RUNNER_SHA256 = "0f5ae21e747119e61fd4a2b0636b5031f729d41841e80470db0c56620d991db0"
UNIDET_OFFICIAL_SCRIPT = T2I_REPO / "UniDet_eval" / "2D_spatial_eval.py"
EXPECTED_UNIDET_SOURCE_SHA256 = "1585ca6043f67de99b561e9e5f356e50d6ddb251c16288f552b32c842c70afee"
UNIDET_BATCH_SIZE = 32
if sha256_file(UNIDET_RUNNER) != EXPECTED_UNIDET_RUNNER_SHA256:
    raise RuntimeError("Memory-safe UniDet runner changed; review it and update the locked hash deliberately")
if sha256_file(UNIDET_OFFICIAL_SCRIPT) != EXPECTED_UNIDET_SOURCE_SHA256:
    raise RuntimeError("Pinned official UniDet scorer source hash mismatch")

def evaluator_env():
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(PROTOCOL.physical_gpu_evaluator)
    env["PYTHONUNBUFFERED"] = "1"
    return env


def run_logged(command: list[str], cwd: Path, log_path: Path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("+", " ".join(command), f"(log: {log_path})")
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, cwd=str(cwd), env=evaluator_env(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        assert process.stdout is not None
        for line in process.stdout:
            log.write(line)
            if "score" in line.lower() or "processed" in line.lower():
                print(line.rstrip())
        return_code = process.wait()
    if return_code:
        tail = "\n".join(log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-80:])
        raise RuntimeError(f"Official scorer failed with exit code {return_code}. Log: {log_path}\n{tail}")


def average_answers(path: Path, expected_count: int = 250) -> float:
    data = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(data, list) or len(data) != expected_count:
        raise RuntimeError(f"Expected {expected_count} official records in {path}; found {len(data) if isinstance(data, list) else type(data)}")
    values = [float(item["answer"]) for item in data]
    if not all(math.isfinite(value) for value in values):
        raise RuntimeError(f"Non-finite official scorer output in {path}")
    return float(np.mean(values))


def category_image_digest(method: str, category: str) -> str:
    stage = ARTIFACT_ROOT / "t2i_compbench" / method / category
    rows = []
    for sidecar in sorted((stage / "sample_metadata").glob("*.json")):
        metadata = json.loads(sidecar.read_text(encoding="utf-8"))
        rows.append((metadata["question_id"], metadata["image_sha256"]))
    if len(rows) != 250:
        raise RuntimeError(f"Generation incomplete: {method}/{category}")
    return canonical_hash(rows)


def run_official_t2i_scoring():
    scores = {}
    for method in METHODS:
        method_scores = {}
        for category in T2I_CATEGORIES:
            stage = ARTIFACT_ROOT / "t2i_compbench" / method / category
            raw_dir = stage / "raw_scorer_outputs"
            raw_dir.mkdir(parents=True, exist_ok=True)
            provenance = {
                "protocol_hash": PROTOCOL_HASH,
                "official_repo_commit": PINS["t2i_compbench"][1],
                "method": method, "category": category,
                "expected_count": 250,
                "image_set_sha256": category_image_digest(method, category),
            }
            if category == "spatial":
                provenance["unidet_memory_adaptation"] = {
                    "batch_size": UNIDET_BATCH_SIZE,
                    "upstream_batch_size": 64,
                    "official_source_sha256": EXPECTED_UNIDET_SOURCE_SHA256,
                    "runner_sha256": EXPECTED_UNIDET_RUNNER_SHA256,
                    "scope": "batch size only; model, weights, preprocessing, detection, and scoring logic unchanged",
                }
            provenance_path = raw_dir / "provenance.json"
            if category == "spatial":
                command = [
                    str(T2I_ENV_PYTHON), str(UNIDET_RUNNER),
                    "--official-script", str(UNIDET_OFFICIAL_SCRIPT),
                    "--expected-source-sha256", EXPECTED_UNIDET_SOURCE_SHA256,
                    "--outpath", str(stage), "--batch-size", str(UNIDET_BATCH_SIZE),
                ]
                cwd = T2I_REPO / "UniDet_eval"
                result_path = stage / "labels" / "annotation_obj_detection_2d" / "vqa_result.json"
            else:
                command = [str(T2I_ENV_PYTHON), "BLIP_vqa.py", "--out_dir", str(stage), "--np_num", "8"]
                cwd = T2I_REPO / "BLIPvqa_eval"
                result_path = stage / "annotation_blip" / "vqa_result.json"
            resume = False
            if provenance_path.exists() and result_path.exists():
                resume = json.loads(provenance_path.read_text()) == provenance
                if resume:
                    average_answers(result_path)
            if not resume:
                run_logged(command, cwd, raw_dir / "official_scorer.log")
                write_json(provenance, provenance_path)
            score = average_answers(result_path)
            shutil.copy2(result_path, raw_dir / "vqa_result.json")
            method_scores[category] = score
            write_json({**provenance, "score": score, "raw_result": str(result_path)}, raw_dir / "score.json")
            gc.collect()
            torch.cuda.empty_cache()
        method_scores["aggregate"] = float(np.mean([method_scores[category] for category in T2I_CATEGORIES]))
        scores[method] = method_scores
        write_json(method_scores, ARTIFACT_ROOT / "t2i_compbench" / method / "final_results.json")
    write_json({"protocol_hash": PROTOCOL_HASH, "subset": "100 prompts / 1,000 images per method", "scores": scores}, ARTIFACT_ROOT / "t2i_compbench" / "final_results.json")
    return scores


T2I_SCORES = run_official_t2i_scoring()
pd.DataFrame(T2I_SCORES).T


+ /media/fezan/ASi/DVLM/steering/aim-flow/.venv/t2i-compbench-py310/bin/python BLIP_vqa.py --out_dir /media/fezan/ASi/DVLM/steering/aim-flow/outputs/rectified_cfgpp_paper_reproduction_t2i_100_seed13/t2i_compbench/cfg/color --np_num 8 (log: /media/fezan/ASi/DVLM/steering/aim-flow/outputs/rectified_cfgpp_paper_reproduction_t2i_100_seed13/t2i_compbench/cfg/color/raw_scorer_outputs/official_scorer.log)
Number of Processed Images: 250
Number of Processed Images: 250
Number of Processed Images: 250
Number of Processed Images: 250
Number of Processed Images: 250
Number of Processed Images: 250
Number of Processed Images: 250
Number of Processed Images: 250
BLIP-VQA score: 0.789811599999999 !
+ /media/fezan/ASi/DVLM/steering/aim-flow/.venv/t2i-compbench-py310/bin/python BLIP_vqa.py --out_dir /media/fezan/ASi/DVLM/steering/aim-flow/outputs/rectified_cfgpp_paper_reproduction_t2i_100_seed13/t2i_compbench/cfg/texture --np_num 8 (log: /media/fezan/ASi/DVLM/steering/aim-flow/outputs/rectified_cfgpp_

,color,texture,spatial,shape,aggregate
cfg,0.789812,0.654442,0.217147,0.580700,0.560525
rectified_cfgpp,0.787298,0.670959,0.204096,0.607231,0.567396


## Final seeded reproduction comparison

All values are official T2I-CompBench outputs for the same 100 prompts and ten initial seeds. Higher is better. Paired prompt-cluster bootstrap intervals average the ten images for each prompt before resampling prompts.


In [11]:
rows = []
for method in METHODS:
    score = T2I_SCORES[method]
    rows.append({
        "Method": METHOD_LABELS[method], "T2I Color": score["color"],
        "T2I Texture": score["texture"], "T2I Spatial": score["spatial"],
        "T2I Shape": score["shape"], "T2I Aggregate": score["aggregate"],
    })
FINAL_RESULTS = pd.DataFrame(rows)
RESULTS_DIR = ARTIFACT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = RESULTS_DIR / f"t2i_compbench_rectified_cfgpp_paper_reproduction_seed{RUN_SEED}.csv"
RESULTS_JSON = RESULTS_DIR / f"t2i_compbench_rectified_cfgpp_paper_reproduction_seed{RUN_SEED}.json"
FINAL_RESULTS.to_csv(RESULTS_CSV, index=False)
write_json(rows, RESULTS_JSON)

def official_result_path(method: str, category: str) -> Path:
    stage = ARTIFACT_ROOT / "t2i_compbench" / method / category
    return (stage / "labels" / "annotation_obj_detection_2d" / "vqa_result.json") if category == "spatial" else (stage / "annotation_blip" / "vqa_result.json")

def prompt_level_scores(method: str, category: str) -> np.ndarray:
    records = json.loads(official_result_path(method, category).read_text(encoding="utf-8"))
    by_qid = {int(record["question_id"]): float(record["answer"]) for record in records}
    if set(by_qid) != set(range(250)):
        raise RuntimeError(f"Official result IDs incomplete: {method}/{category}")
    values = np.array([by_qid[qid] for qid in range(250)], dtype=np.float64)
    return values.reshape(PROTOCOL.prompts_per_category, PROTOCOL.samples_per_prompt).mean(axis=1)

bootstrap_rng = np.random.default_rng(PROTOCOL.selection_seed)
category_statistics, all_prompt_deltas = [], []
for category in T2I_CATEGORIES:
    cfg_values = prompt_level_scores("cfg", category)
    rect_values = prompt_level_scores("rectified_cfgpp", category)
    deltas = rect_values - cfg_values
    all_prompt_deltas.extend(deltas.tolist())
    boot = deltas[bootstrap_rng.integers(0, len(deltas), size=(20_000, len(deltas)))].mean(axis=1)
    category_statistics.append({
        "Category": category, "CFG": float(cfg_values.mean()), "Rectified-CFG++": float(rect_values.mean()),
        "Rectified minus CFG": float(deltas.mean()),
        "Prompt-cluster bootstrap 95% low": float(np.quantile(boot, 0.025)),
        "Prompt-cluster bootstrap 95% high": float(np.quantile(boot, 0.975)),
        "Distinct prompts": len(deltas), "Images per prompt": PROTOCOL.samples_per_prompt,
    })
all_prompt_deltas = np.asarray(all_prompt_deltas, dtype=np.float64)
aggregate_boot = all_prompt_deltas[bootstrap_rng.integers(0, len(all_prompt_deltas), size=(20_000, len(all_prompt_deltas)))].mean(axis=1)
category_statistics.append({
    "Category": "aggregate",
    "CFG": float(FINAL_RESULTS.loc[FINAL_RESULTS["Method"] == METHOD_LABELS["cfg"], "T2I Aggregate"].iloc[0]),
    "Rectified-CFG++": float(FINAL_RESULTS.loc[FINAL_RESULTS["Method"] == METHOD_LABELS["rectified_cfgpp"], "T2I Aggregate"].iloc[0]),
    "Rectified minus CFG": float(all_prompt_deltas.mean()),
    "Prompt-cluster bootstrap 95% low": float(np.quantile(aggregate_boot, 0.025)),
    "Prompt-cluster bootstrap 95% high": float(np.quantile(aggregate_boot, 0.975)),
    "Distinct prompts": len(all_prompt_deltas), "Images per prompt": PROTOCOL.samples_per_prompt,
})
STATISTICS = pd.DataFrame(category_statistics)
STATISTICS.to_csv(RESULTS_DIR / "paired_prompt_cluster_statistics.csv", index=False)
write_json(category_statistics, RESULTS_DIR / "paired_prompt_cluster_statistics.json")

paper_reference = {
    "cfg": {"color": 0.7658, "shape": 0.5698, "texture": 0.7270, "spatial": 0.3199},
    "rectified_cfgpp": {"color": 0.8041, "shape": 0.5778, "texture": 0.7362, "spatial": 0.3306},
    "source": "Rectified-CFG++ NeurIPS 2025 paper, Table 2, SD3 rows",
    "comparability_note": "Reference only: this run is a deterministic 100-prompt subset.",
}
write_json(paper_reference, RESULTS_DIR / "paper_table2_sd3_reference.json")

package_names = ["torch", "torchvision", "diffusers", "transformers", "accelerate", "huggingface-hub", "safetensors", "sentencepiece", "protobuf", "numpy", "Pillow", "pandas", "tqdm"]
package_versions = {}
for name in package_names:
    try:
        package_versions[name] = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        package_versions[name] = None
compliance = {
    "methods_exactly_cfg_and_rectified": METHODS == ("cfg", "rectified_cfgpp"),
    "cfg_omega_4_5": PROTOCOL.cfg_guidance_scale == 4.5,
    "rectified_lambda_7_0": PROTOCOL.rectified_guidance_scale == 7.0,
    "sd3_revision_pinned": info.sha == PROTOCOL.model_revision,
    "resolution_1024": (PROTOCOL.width, PROTOCOL.height) == (1024, 1024),
    "nfe_28": PROTOCOL.num_inference_steps == 28,
    "full_t5_exact_equivalence": bool(index_record["singleton_cfg_exact_equivalence_audit"]["passed"]),
    "official_rectified_commit": git_output(REPO_PATHS["rectified_cfgpp"], "rev-parse", "HEAD") == PINS["rectified_cfgpp"][1],
    "official_t2i_commit": git_output(T2I_REPO, "rev-parse", "HEAD") == PINS["t2i_compbench"][1],
    "balanced_selected_subset": subset_audit["counts"] == {category: PROTOCOL.prompts_per_category for category in T2I_CATEGORIES},
    "unidet_memory_adapter_locked": (
        UNIDET_BATCH_SIZE == 32
        and sha256_file(UNIDET_RUNNER) == EXPECTED_UNIDET_RUNNER_SHA256
        and sha256_file(UNIDET_OFFICIAL_SCRIPT) == EXPECTED_UNIDET_SOURCE_SHA256
    ),
    "known_scope_boundary": reproduction_scope["disclosure_boundary"],
}
if not all(value is True for key, value in compliance.items() if key != "known_scope_boundary"):
    raise RuntimeError(f"Compliance gate failed: {compliance}")
write_json(compliance, RESULTS_DIR / "compliance_gate.json")
reproducibility = {
    **protocol_record, "subset_audit": subset_audit,
    "full_t5_cache_index": str(EMBEDDING_INDEX), "full_t5_cache_index_sha256": sha256_file(EMBEDDING_INDEX),
    "worker_source": str(WORKER_PATH), "worker_source_sha256": sha256_file(WORKER_PATH),
    "released_pipeline_algorithm_audit": released_pipeline_algorithm_audit,
    "generation_audit": generation_audit, "generation_package_versions": package_versions,
    "official_scorer_freeze": str(T2I_LOCK_DIR / "t2i_compbench_pip_freeze.txt"),
    "official_scorer_freeze_sha256": sha256_file(T2I_LOCK_DIR / "t2i_compbench_pip_freeze.txt"),
    "unidet_memory_adapter": {
        "runner": str(UNIDET_RUNNER), "runner_sha256": sha256_file(UNIDET_RUNNER),
        "official_source": str(UNIDET_OFFICIAL_SCRIPT), "official_source_sha256": sha256_file(UNIDET_OFFICIAL_SCRIPT),
        "upstream_batch_size": 64, "executed_batch_size": UNIDET_BATCH_SIZE,
    },
    "intentional_deviation_from_full_benchmark": "25 prompts/category rather than approximately 300; ten official metric samples per prompt retained.",
    "paper_reference": paper_reference, "compliance_gate": compliance,
    "result_files": {"csv": str(RESULTS_CSV), "json": str(RESULTS_JSON), "statistics_csv": str(RESULTS_DIR / "paired_prompt_cluster_statistics.csv")},
}
write_json(reproducibility, RESULTS_DIR / "reproducibility_manifest.json")
display(FINAL_RESULTS.style.format({column: "{:.6f}" for column in FINAL_RESULTS.columns if column != "Method"}))
display(STATISTICS.style.format({
    "CFG": "{:.6f}", "Rectified-CFG++": "{:.6f}", "Rectified minus CFG": "{:+.6f}",
    "Prompt-cluster bootstrap 95% low": "{:+.6f}", "Prompt-cluster bootstrap 95% high": "{:+.6f}",
}))
print(f"Results: {RESULTS_CSV}")
print(f"Reproducibility manifest: {RESULTS_DIR / 'reproducibility_manifest.json'}")


,Method,T2I Color,T2I Texture,T2I Spatial,T2I Shape,T2I Aggregate
0,CFG (ω=4.5),0.789812,0.654442,0.217147,0.580700,0.560525
1,Rectified-CFG++ (λ=7.0),0.787298,0.670959,0.204096,0.607231,0.567396


,Category,CFG,Rectified-CFG++,Rectified minus CFG,Prompt-cluster bootstrap 95% low,Prompt-cluster bootstrap 95% high,Distinct prompts,Images per prompt
0,color,0.789812,0.787298,-0.002514,-0.020589,+0.018975,25,10
1,texture,0.654442,0.670959,+0.016517,-0.012571,+0.044942,25,10
2,spatial,0.217147,0.204096,-0.013051,-0.062948,+0.029780,25,10
3,shape,0.580700,0.607231,+0.026531,+0.000556,+0.056979,25,10
4,aggregate,0.560525,0.567396,+0.006871,-0.009647,+0.023067,100,10


Results: /media/fezan/ASi/DVLM/steering/aim-flow/outputs/rectified_cfgpp_paper_reproduction_t2i_100_seed13/results/t2i_compbench_rectified_cfgpp_paper_reproduction_seed13.csv
Reproducibility manifest: /media/fezan/ASi/DVLM/steering/aim-flow/outputs/rectified_cfgpp_paper_reproduction_t2i_100_seed13/results/reproducibility_manifest.json
